# 03 Run Zero-Shot Mutant Design

This notebook is a lightweight orchestrator around `run_zero_shot_mutant_design_workflow`, with heavy logic moved into `src/agentic_protein_design/workflows/run_zero_shot_mutant_design/`.


In [1]:
from pathlib import Path
import sys
import pandas as pd

# Section 1: Resolve repo root from either project root or notebooks/ cwd.
cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd

# Section 2: Add repo and src roots to sys.path for imports.
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

print("repo_root:", repo_root)
print("src_root:", src_root)


repo_root: /Users/charmainechia/Documents/projects/agentic-protein-design
src_root: /Users/charmainechia/Documents/projects/agentic-protein-design/src


In [2]:
from pprint import pprint

from agentic_protein_design.workflows.run_zero_shot_mutant_design.config import build_user_inputs
from agentic_protein_design.workflows.run_zero_shot_mutant_design.workflow import run_zero_shot_mutant_design_workflow


In [10]:
# Section 1: Build compact user config for zero-shot design workflow.
user_inputs = build_user_inputs(
    root_key='ECOHARVEST', # 'ppk2-hmo', # "examples",
    output_data_subfolder="RML-propeptide-mature_R0",
    filename_prefix="RML-propeptide-mature_",
    output_filename_suffix="_distal", # "", #
    ligand='SucroseOleate',# "S82",
    wt_sequence="",
    marginal_type="masked",
    score_types_to_run={
        "plm_llr": ["esm2-650m", "esmc-600m", "poet2"],
        "proteinmpnn": [],
        "conservation": [],
        "stability_ddg": ["SPURS"],
        "structure_annotations": ['distance'] # ["distance", "residue_properties"],
    },
    col_constraints='((LLR_avg>0)|(ddg_SPURS<0))&(min_dist_to_lig>8)', # '(LLR_avg>0)&(ddg_SPURS<0)', #
    pos_to_exclude=[],
    allowed_positions=[],
    mutations_to_exclude=[],
    top_n=50,
    max_num_mut_per_pos=3,
)

# Section 2: Optional filename hooks.
user_inputs["wt_sequence_filename"] = ""
user_inputs["candidate_sequences_filename"] = ""
user_inputs["conservation_filename"] = ""
user_inputs["structure_filename"] = ""
user_inputs["ligand_filename"] = ""
user_inputs["llr_cache_vect_filename_prefix"] = ""

pprint(user_inputs)


{'allowed_mut_aas': [],
 'allowed_positions': [],
 'candidate_sequences_filename': '',
 'col_constraints': '((LLR_avg>0)|(ddg_SPURS<0))&(min_dist_to_lig>8)',
 'conservation_filename': '',
 'dist_to_lig_filter_direction': 'le',
 'dist_to_lig_thres': None,
 'filename_prefix': 'RML-propeptide-mature_',
 'ligand': 'SucroseOleate',
 'ligand_filename': '',
 'llr_cache_vect_filename_prefix': '',
 'marginal_type': 'masked',
 'max_num_mut_per_pos': 3,
 'mutations_to_exclude': [],
 'output_data_subfolder': 'RML-propeptide-mature_R0',
 'output_filename_suffix': '_distal',
 'plm_models': ['esm2-650m', 'esmc-600m'],
 'pos_to_exclude': [],
 'root_key': 'ECOHARVEST',
 'score_types_to_run': {'conservation': [],
                        'plm_llr': ['esm2-650m', 'esmc-600m', 'poet2'],
                        'plm_meanpll': [],
                        'proteinmpnn': [],
                        'spurs': [],
                        'stability_ddg': ['SPURS'],
                        'structure_annotations':

In [11]:
# Section 1: Run the overall workflow driver (prototype).
import pandas as pd

result = run_zero_shot_mutant_design_workflow(user_inputs, repo_root=repo_root)
print("status:", result.get("status"))

# Section 2: Display final shortlisted score table + size.
shortlist_df = pd.DataFrame(result.get("shortlist_records", []))
print("\nshortlist_df.shape:", shortlist_df.shape)
print("shortlist_df.columns:", list(shortlist_df.columns))
print("missing_requested_columns:", result.get("compiled_scores_missing_columns", []))
shortlist_df


[Step] plm_scoring: START (up_to_date)
- esm2-650m LLR vect: FOUND
  target: /Users/charmainechia/Documents/projects/ECOHARVEST/encodings/LLR/RML-propeptide-mature_esm2-650m_LLR-masked_vect.csv
- esmc-600m LLR vect: FOUND
  target: /Users/charmainechia/Documents/projects/ECOHARVEST/encodings/LLR/RML-propeptide-mature_esmc-600m_LLR-masked_vect.csv
- poet2 LLR vect: FOUND
  target: /Users/charmainechia/Documents/projects/ECOHARVEST/encodings/LLR/RML-propeptide-mature_poet2_LLR_vect.csv
[Step] plm_scoring: COMPLETE
[Step] spurs_scoring: START (up_to_date)
- SPURS scores: FOUND
  target: /Users/charmainechia/Documents/projects/ECOHARVEST/stability/ddg/RML-propeptide-mature_SPURS_ddg_vect.csv
[Step] spurs_scoring: COMPLETE
[Step] structure_annotations: START (up_to_date)
- structure annotations (distance): FOUND
  target: /Users/charmainechia/Documents/projects/ECOHARVEST/pdb/structure_csv/RML-propeptide-mature_SucroseOleate_distance.csv
[Step] structure_annotations: COMPLETE
[Compiled Scor

,resnum,mutations,LLR_esm2-650m,LLR_esmc-600m,LLR_poet2,ddg_SPURS,min_dist_to_lig,wt_aa,mt_aa,LLR_consensus,LLR_avg,selection_rank
0,140,G140S,4.7232,5.1628,3.7752,-0.740708,24.121,G,S,3,4.553733,1
1,201,F201L,3.5426,4.6285,3.8248,-0.158912,28.223,F,L,3,3.998633,2
2,131,Y131T,5.8778,3.4441,2.2003,0.657227,10.563,Y,T,3,3.840733,3
3,140,G140D,3.3303,2.7659,4.5514,0.409680,24.121,G,D,3,3.549200,4
4,201,F201I,3.1957,3.6252,2.7572,0.240288,28.223,F,I,3,3.192700,5
5,201,F201V,3.1675,3.5389,2.6808,0.380092,28.223,F,V,3,3.129067,6
6,232,E232V,3.2851,4.4504,1.4775,0.340986,28.124,E,V,3,3.071000,7
7,220,T220L,3.2020,2.9810,2.9752,-1.935661,9.454,T,L,3,3.052733,8
8,176,T176A,3.5151,3.1854,2.3111,-0.665416,16.630,T,A,3,3.003867,9
9,233,E233P,3.7547,2.8895,2.2130,0.227152,30.882,E,P,3,2.952400,10
